# Therapeutic Optimization — Colab Runner

This notebook is intentionally lightweight. All reusable logic lives in `src/therapeutic_optimization/`.

Pipeline: **T1 → UP1 → T2 → S1 → R1 → UB2 → R2**.

Before publishing, replace the placeholder GitHub URL below with the real repository URL.


In [1]:
PROTEIN_ID = "my_protein"
WT_SEQUENCE = """PTAPPYDSLLVFDYEGGSGSGSGASRLNFGDDIPSALRIAKKKRWNSIEERRIHQESELHSYLSRLIAAERERELEECQRNHEGDEDDSHVRAQQACIEAKHDKYMADMDELFSQVDEKRKKRDIPDYLCGKISFELMREPCITPSGITYDRKDIEEHLQRVGHFDPVTRSPLTQEQLIPNLAMKEVIDAFISENGWVEDY"""


## Hyperparameters

`single` changes one selected lysine at a time. `combinatorial` generates orders 1 through `MAX_COMBINATION_ORDER` using every replacement amino acid listed in `REPLACEMENT_AAS`.


In [2]:
UBI_THRESHOLD = 0.36 ## least conservative true positive, false negative result F2 EUP presentation paper

MUTATION_MODE = "single"            # "single" or "combinatorial"
REPLACEMENT_AAS = ("R",)            # e.g. ("A", "R", "Q")
MAX_COMBINATION_ORDER = 4
MAX_VARIANTS = 5000

# S1 structural-preservation heuristic gates
GLOBAL_CA_RMSD_MAX = 1.0
LOCAL_MEAN_CA_DISPLACEMENT_MAX = 1.5
MUTATION_CA_DISPLACEMENT_MAX = 2.0
CONTACT_CHANGE_FRACTION_MAX = 0.10
MIN_MEAN_PLDDT = 70.0


## Runtime, dependencies, imports

Use a **GPU runtime**. EUP uses ESM2-3B and this implementation intentionally fails rather than silently falling back to a very slow CPU path.


In [3]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/juliaevizza/therapeutic_optimization.git"
REPO_DIR = Path("/content/therapeutic_optimization")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository ready:", REPO_DIR)

# Run this after REPO_URL is configured and the repository has been cloned.
if REPO_DIR.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[eup]"], check=True)

    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)

    if shutil.which("colabfold_batch") is None:
        print("colabfold_batch not found. Installing the current CUDA 12 ColabFold stack...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "colabfold[alphafold,openmm]", "jax[cuda12]", "openmm[cuda12]"
        ], check=True)

    subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "check_environment.py")], check=False)


Repository ready: /content/therapeutic_optimization
colabfold_batch not found. Installing the current CUDA 12 ColabFold stack...


## Choose where run outputs live

Set `SAVE_TO_DRIVE = True` if you want the run to survive Colab shutdown. The package code still runs from GitHub; only `storage/` outputs go into this run directory.


In [4]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/therapeutic_optimization_run")
else:
    RUN_ROOT = Path("/content/therapeutic_optimization_run")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Run root:", RUN_ROOT)


Mounted at /content/drive
Run root: /content/drive/MyDrive/therapeutic_optimization_run


## Build the workflow


In [5]:
import sys
if 'therapeutic_optimization' in sys.modules:
    del sys.modules['therapeutic_optimization']
if "/content/therapeutic_optimization/src" not in sys.path:
    sys.path.insert(0, "/content/therapeutic_optimization/src")

from therapeutic_optimization.config import (
    MutationConfig,
    PredictorConfig,
    StructuralThresholds,
    WorkflowConfig,
)
from therapeutic_optimization.workflow import OptimizationWorkflow
from pathlib import Path

config = WorkflowConfig(
    mutation=MutationConfig(
        threshold=UBI_THRESHOLD,
        mode=MUTATION_MODE,
        replacement_aas=REPLACEMENT_AAS,
        max_combination_order=MAX_COMBINATION_ORDER,
        max_variants=MAX_VARIANTS,
    ),
    ubiquitination=PredictorConfig(
        name="eup",
        threshold=UBI_THRESHOLD,
        eup_repo_dir=Path("/content/external/EUP"),
        model_cache_dir=Path("/content/huggingface"),
    ),
    structural_thresholds=StructuralThresholds(
        global_ca_rmsd_max=GLOBAL_CA_RMSD_MAX,
        local_mean_ca_displacement_max=LOCAL_MEAN_CA_DISPLACEMENT_MAX,
        mutation_ca_displacement_max=MUTATION_CA_DISPLACEMENT_MAX,
        contact_change_fraction_max=CONTACT_CHANGE_FRACTION_MAX,
        min_mean_plddt=MIN_MEAN_PLDDT,
    ),
)

workflow = OptimizationWorkflow(RUN_ROOT, config)

## T1: input → WT FASTA


In [6]:
t1 = workflow.T1(WT_SEQUENCE, PROTEIN_ID)
t1

{'protein_id': 'my_protein',
 'sequence_length': 201,
 'wt_fasta': '/content/drive/MyDrive/therapeutic_optimization_run/storage/inputs/wt_input.fasta',
 'created_at_utc': '2026-09-03T00:02:11.604076+00:00'}

## UP1: WT ubiquitination prediction


In [7]:
up1 = workflow.UP1()
display(up1)


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/58.0k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/582 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t36_3B_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,variant_id,protein_id,predictor,lysine_position,site,sequence_context,probability,threshold,is_positive
0,WT,my_protein,EUP,153,K153,ITPSGITYDRKDIEEHLQRVG,0.609378,0.36,True
1,WT,my_protein,EUP,119,K119,MDELFSQVDEKRKKRDIPDYL,0.547615,0.36,True
2,WT,my_protein,EUP,185,K185,QEQLIPNLAMKEVIDAFISEN,0.524864,0.36,True
3,WT,my_protein,EUP,101,K101,VRAQQACIEAKHDKYMADMDE,0.499223,0.36,True
4,WT,my_protein,EUP,104,K104,QQACIEAKHDKYMADMDELFS,0.460844,0.36,True
5,WT,my_protein,EUP,132,K132,KRDIPDYLCGKISFELMREPC,0.400665,0.36,True
6,WT,my_protein,EUP,122,K122,LFSQVDEKRKKRDIPDYLCGK,0.332685,0.36,False
7,WT,my_protein,EUP,43,K43,IPSALRIAKKKRWNSIEERRI,0.302615,0.36,False
8,WT,my_protein,EUP,41,K41,DDIPSALRIAKKKRWNSIEER,0.288181,0.36,False
9,WT,my_protein,EUP,42,K42,DIPSALRIAKKKRWNSIEERR,0.245417,0.36,False


## T2: generate mutant FASTAs


In [8]:
t2 = workflow.T2(up1)
print(f"Generated {len(t2)} mutant(s).")
display(t2)


Generated 56 mutant(s).


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error
0,K101R,K101R,1,K101,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
1,K104R,K104R,1,K104,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
2,K119R,K119R,1,K119,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
3,K132R,K132R,1,K132,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
4,K153R,K153R,1,K153,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
5,K185R,K185R,1,K185,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
6,K101R__K104R,K101R;K104R,2,K101;K104,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
7,K101R__K119R,K101R;K119R,2,K101;K119,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
8,K101R__K132R,K101R;K132R,2,K101;K132,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None
9,K101R__K153R,K101R;K153R,2,K101;K153,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None


# P1: perplexity

In [ ]:
from therapeutic_optimization.esm2_analysis import run_esm2_analysis

t2 = workflow.T2(up1)
esm2_results = run_esm2_analysis(t2, workflow.paths)

## S1: structure prediction + preservation screen

This is the expensive structure stage. Every T2 mutant is predicted and compared with WT.


In [9]:
s1_metrics, s1_conserved = workflow.S1(t2, predict_structures=True)
print(f"Structurally conserved: {len(s1_conserved)} / {len(s1_metrics)}")
display(s1_metrics)
display(s1_conserved)


Structurally conserved: 16 / 56


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error,global_ca_rmsd,...,mean_centroid_distance_change,mean_residue_volume_change,per_residue_csv,displacement_plot,wt_structure,mutant_structure,analysis_status,analysis_error,structure_pass,structural_preservation_score
0,K101R,K101R,1,K101,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.172226,...,0.170307,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.807547
1,K104R,K104R,1,K104,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.043241,...,0.128237,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.772437
2,K119R,K119R,1,K119,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.666752,...,-0.034152,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.851707
3,K132R,K132R,1,K132,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.197168,...,-0.070995,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.775455
4,K153R,K153R,1,K153,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,6.884772,...,0.493323,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.594432
5,K185R,K185R,1,K185,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.616363,...,-0.044704,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.655386
6,K101R__K104R,K101R;K104R,2,K101;K104,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.176593,...,0.313578,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.779672
7,K101R__K119R,K101R;K119R,2,K101;K119,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.035020,...,0.047244,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.795599
8,K101R__K132R,K101R;K132R,2,K101;K132,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.776080,...,0.137578,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.840352
9,K101R__K153R,K101R;K153R,2,K101;K153,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,1.327859,...,0.146115,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,False,0.772675


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error,global_ca_rmsd,...,mean_centroid_distance_change,mean_residue_volume_change,per_residue_csv,displacement_plot,wt_structure,mutant_structure,analysis_status,analysis_error,structure_pass,structural_preservation_score
18,K132R__K153R,K132R;K153R,2,K132;K153,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.584943,...,0.099134,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.862666
2,K119R,K119R,1,K119,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.666752,...,-0.034152,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.851707
23,K101R__K104R__K153R,K101R;K104R;K153R,3,K101;K104;K153,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.759983,...,0.115680,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.847840
21,K101R__K104R__K119R,K101R;K104R;K119R,3,K101;K104;K119,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.679258,...,0.112782,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.845673
8,K101R__K132R,K101R;K132R,2,K101;K132,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.776080,...,0.137578,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.840352
36,K104R__K153R__K185R,K104R;K153R;K185R,3,K104;K153;K185,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.715787,...,-0.026064,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.839838
50,K101R__K132R__K153R__K185R,K101R;K132R;K153R;K185R,4,K101;K132;K153;K185,R;R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.760202,...,0.069703,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.838986
12,K104R__K132R,K104R;K132R,2,K104;K132,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.846244,...,0.171561,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.820977
13,K104R__K153R,K104R;K153R,2,K104;K153,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.845833,...,0.057438,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,True,0.819667
24,K101R__K104R__K185R,K101R;K104R;K185R,3,K101;K104;K185,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,0.810393,...,0.144027,13.0,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,Non

## R1: record structural dropouts


In [10]:
r1 = workflow.R1(t2, s1_metrics)
display(r1)


,variant_id,mutation_spec,mutation_count,source_sites,replacement_aas,sequence_length,fasta_path,status,error,analysis_status,analysis_error,structure_pass,structural_preservation_score,global_ca_rmsd,local_mean_ca_displacement,mutation_ca_displacement_max,global_contact_change_fraction,mutant_mean_plddt,dropped_after_S1,R1_status
0,K132R__K153R,K132R;K153R,2,K132;K153,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.862666,0.584943,0.285371,0.345519,0.007663,82.561990,False,ADVANCE_TO_UB2
1,K119R,K119R,1,K119,R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.851707,0.666752,0.291030,0.428449,0.006386,82.795970,False,ADVANCE_TO_UB2
2,K101R__K104R__K153R,K101R;K104R;K153R,3,K101;K104;K153,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.847840,0.759983,0.230598,0.393218,0.006394,82.633184,False,ADVANCE_TO_UB2
3,K101R__K104R__K119R,K101R;K104R;K119R,3,K101;K104;K119,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.845673,0.679258,0.336993,0.468139,0.006394,82.932786,False,ADVANCE_TO_UB2
4,K101R__K132R,K101R;K132R,2,K101;K132,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.840352,0.776080,0.320692,0.459594,0.003836,83.077960,False,ADVANCE_TO_UB2
5,K104R__K153R__K185R,K104R;K153R;K185R,3,K104;K153;K185,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.839838,0.715787,0.344651,0.534151,0.003841,82.471095,False,ADVANCE_TO_UB2
6,K101R__K132R__K153R__K185R,K101R;K132R;K153R;K185R,4,K101;K132;K153;K185,R;R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.838986,0.760202,0.342165,0.448165,0.005122,82.673682,False,ADVANCE_TO_UB2
7,K104R__K132R,K104R;K132R,2,K104;K132,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.820977,0.846244,0.368082,0.614930,0.003846,82.366617,False,ADVANCE_TO_UB2
8,K104R__K153R,K104R;K153R,2,K104;K153,R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.819667,0.845833,0.422240,0.562202,0.005128,82.818856,False,ADVANCE_TO_UB2
9,K101R__K104R__K185R,K101R;K104R;K185R,3,K101;K104;K185,R;R;R,201,/content/drive/MyDrive/therapeutic_optimizatio...,PASS,None,PASS,None,True,0.819597,0.810393,0.443815,0.571654,0.007663,83.234577,False,ADVANCE_TO_UB2


## UB2: rerun ubiquitination prediction on S1 survivors


In [11]:
ub2 = workflow.UB2(s1_conserved)
display(ub2)


,variant_id,protein_id,predictor,lysine_position,site,sequence_context,probability,threshold,is_positive
0,K132R__K153R,K132R__K153R,EUP,119,K119,MDELFSQVDEKRKKRDIPDYL,0.520525,0.36,True
1,K132R__K153R,K132R__K153R,EUP,185,K185,QEQLIPNLAMKEVIDAFISEN,0.495605,0.36,True
2,K132R__K153R,K132R__K153R,EUP,101,K101,VRAQQACIEAKHDKYMADMDE,0.487270,0.36,True
3,K132R__K153R,K132R__K153R,EUP,104,K104,QQACIEAKHDKYMADMDELFS,0.457296,0.36,True
4,K132R__K153R,K132R__K153R,EUP,122,K122,LFSQVDEKRKKRDIPDYLCGR,0.309721,0.36,False
...,...,...,...,...,...,...,...,...,...
126,K101R__K119R__K153R,K101R__K119R__K153R,EUP,122,K122,LFSQVDERRKKRDIPDYLCGK,0.309491,0.36,False
127,K101R__K119R__K153R,K101R__K119R__K153R,EUP,43,K43,IPSALRIAKKKRWNSIEERRI,0.287358,0.36,False
128,K101R__K119R__K153R,K101R__K119R__K153R,EUP,41,K41,DDIPSALRIAKKKRWNSIEER,0.272672,0.36,False
129,K101R__K119R__K153R,K101R__K119R__K153R,EUP,42,K42,DIPSALRIAKKKRWNSIEERR,0.235109,0.36,False


## R2: final ranking


In [12]:
r2_all, optimized, needs_more = workflow.R2(up1, ub2, s1_conserved)

print("OPTIMIZED")
display(optimized)

print("NEEDS FURTHER OPTIMIZATION")
display(needs_more)

print("ALL RANKED")
display(r2_all)


OPTIMIZED


,final_rank,variant_id,mutation_spec,mutation_count,structural_preservation_score,wt_positive_count,mutant_positive_count,targeted_wt_probability_sum,wt_positive_burden,mutant_positive_burden,ubiquitination_burden_reduction,ubiquitination_burden_reduction_fraction,removed_wt_sites,remaining_wt_positive_sites,new_positive_sites,new_positive_count,R2_group


NEEDS FURTHER OPTIMIZATION


,final_rank,variant_id,mutation_spec,mutation_count,structural_preservation_score,wt_positive_count,mutant_positive_count,targeted_wt_probability_sum,wt_positive_burden,mutant_positive_burden,ubiquitination_burden_reduction,ubiquitination_burden_reduction_fraction,removed_wt_sites,remaining_wt_positive_sites,new_positive_sites,new_positive_count,R2_group
0,1,K101R__K119R__K153R__K185R,K101R;K119R;K153R;K185R,4,0.776114,6,2,2.181079,3.042589,0.847840,2.194749,0.721343,K101;K119;K153;K185,K104;K132,,0,needs_further_optimization
1,2,K101R__K132R__K153R__K185R,K101R;K132R;K153R;K185R,4,0.838986,6,2,2.034130,3.042589,0.965246,2.077343,0.682755,K101;K132;K153;K185,K104;K119,,0,needs_further_optimization
2,3,K101R__K104R__K119R__K132R,K101R;K104R;K119R;K132R,4,0.795780,6,2,1.908347,3.042589,1.061210,1.981379,0.651215,K101;K104;K119;K132,K153;K185,,0,needs_further_optimization
3,4,K101R__K119R__K153R,K101R;K119R;K153R,3,0.767060,6,3,1.656216,3.042589,1.352576,1.690013,0.555452,K101;K119;K153,K104;K132;K185,,0,needs_further_optimization
4,5,K101R__K153R__K185R,K101R;K153R;K185R,3,0.807777,6,3,1.633465,3.042589,1.354385,1.688204,0.554858,K101;K153;K185,K104;K119;K132,,0,needs_further_optimization
5,6,K101R__K104R__K153R,K101R;K104R;K153R,3,0.847840,6,3,1.569445,3.042589,1.366730,1.675859,0.550800,K101;K104;K153,K119;K132;K185,,0,needs_further_optimization
6,7,K104R__K153R__K185R,K104R;K153R;K185R,3,0.839838,6,3,1.595086,3.042589,1.412334,1.630255,0.535812,K104;K153;K185,K101;K119;K132,,0,needs_further_optimization
7,8,K119R__K132R__K153R,K119R;K132R;K153R,3,0.805952,6,3,1.557658,3.042589,1.436063,1.606525,0.528013,K119;K132;K153,K101;K104;K185,,0,needs_further_optimization
8,9,K101R__K104R__K119R,K101R;K104R;K119R,3,0.845673,6,3,1.507681,3.042589,1.445174,1.597415,0.525018,K101;K104;K119,K132;K153;K185,,0,needs_further_optimization
9,10,K101R__K104R__K185R,K101R;K104R;K185R,3,0.819597,6,3,1.484930,3.042589,1.455381,1.587208,0.521664,K101;K104;K185,K119;K132;K153,,0,needs_further_optimization


ALL RANKED


,final_rank,variant_id,mutation_spec,mutation_count,structural_preservation_score,wt_positive_count,mutant_positive_count,targeted_wt_probability_sum,wt_positive_burden,mutant_positive_burden,ubiquitination_burden_reduction,ubiquitination_burden_reduction_fraction,removed_wt_sites,remaining_wt_positive_sites,new_positive_sites,new_positive_count,R2_group
0,1,K101R__K119R__K153R__K185R,K101R;K119R;K153R;K185R,4,0.776114,6,2,2.181079,3.042589,0.847840,2.194749,0.721343,K101;K119;K153;K185,K104;K132,,0,needs_further_optimization
1,2,K101R__K132R__K153R__K185R,K101R;K132R;K153R;K185R,4,0.838986,6,2,2.034130,3.042589,0.965246,2.077343,0.682755,K101;K132;K153;K185,K104;K119,,0,needs_further_optimization
2,3,K101R__K104R__K119R__K132R,K101R;K104R;K119R;K132R,4,0.795780,6,2,1.908347,3.042589,1.061210,1.981379,0.651215,K101;K104;K119;K132,K153;K185,,0,needs_further_optimization
3,4,K101R__K119R__K153R,K101R;K119R;K153R,3,0.767060,6,3,1.656216,3.042589,1.352576,1.690013,0.555452,K101;K119;K153,K104;K132;K185,,0,needs_further_optimization
4,5,K101R__K153R__K185R,K101R;K153R;K185R,3,0.807777,6,3,1.633465,3.042589,1.354385,1.688204,0.554858,K101;K153;K185,K104;K119;K132,,0,needs_further_optimization
5,6,K101R__K104R__K153R,K101R;K104R;K153R,3,0.847840,6,3,1.569445,3.042589,1.366730,1.675859,0.550800,K101;K104;K153,K119;K132;K185,,0,needs_further_optimization
6,7,K104R__K153R__K185R,K104R;K153R;K185R,3,0.839838,6,3,1.595086,3.042589,1.412334,1.630255,0.535812,K104;K153;K185,K101;K119;K132,,0,needs_further_optimization
7,8,K119R__K132R__K153R,K119R;K132R;K153R,3,0.805952,6,3,1.557658,3.042589,1.436063,1.606525,0.528013,K119;K132;K153,K101;K104;K185,,0,needs_further_optimization
8,9,K101R__K104R__K119R,K101R;K104R;K119R,3,0.845673,6,3,1.507681,3.042589,1.445174,1.597415,0.525018,K101;K104;K119,K132;K153;K185,,0,needs_further_optimization
9,10,K101R__K104R__K185R,K101R;K104R;K185R,3,0.819597,6,3,1.484930,3.042589,1.455381,1.587208,0.521664,K101;K104;K185,K119;K132;K153,,0,needs_further_optimization


## Inspect outputs


In [13]:
print("Tables:")
for path in sorted((RUN_ROOT / "storage" / "tables").glob("*.csv")):
    print(" -", path)

print("Mutant FASTAs:", len(list((RUN_ROOT / "storage" / "mutants" / "fastas").glob("*.fasta"))))
print("WT structure directory:", RUN_ROOT / "storage" / "structures" / "wt")
print("Mutant structure directory:", RUN_ROOT / "storage" / "structures" / "mutants")


Tables:
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R1_structural_screen.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R2_all_ranked.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R2_needs_further_optimization.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/R2_optimized.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/S1_structural_metrics.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/S1_structurally_conserved.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/T2_mutation_manifest.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/UB2_mutant_ubiquitination.csv
 - /content/drive/MyDrive/therapeutic_optimization_run/storage/tables/UP1_wt_ubiquitination.csv
Mutant FASTAs: 56
WT structure directory: /content/drive/MyDrive/therapeutic_optimization_run/storage/structures/wt
Mutant structur